# Turbofan RUL Tahmini — Bölüm 6
## Streamlit arayüzünü Colab'da çalıştırma (Aşama 13 testi)

Bu notebook:
1. Uygulama dosyalarını Drive'dan alır,
2. **Gerçek final modeli (LightGBM) uygulama koduyla test eder** (test RMSE'nin Bölüm 5 ile aynı çıktığını doğrular),
3. Streamlit'i başlatır ve geçici bir **genel bağlantı (link)** oluşturur.

**Hazırlık:** `streamlit_app.zip` dosyasını Drive'daki `BTK_RUL` klasörüne yükleyin. Zip'i açmanıza gerek yok; ya da `streamlit_app` klasörünü olduğu gibi yükleyebilirsiniz.

> ℹ️ Link **yalnızca bu Colab oturumu açıkken** çalışır. Kalıcı ve herkese açık link için Aşama 14'te Streamlit Community Cloud'a deploy edeceğiz.

### 1. Uygulama dosyalarını hazırla

In [ ]:
import os, shutil, zipfile, sys
DRIVE_DIR = "/content/drive/MyDrive/BTK_RUL"
APP_PARENT = "/content"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:                                   # Colab dışında test için
    DRIVE_DIR, APP_PARENT = os.getcwd(), os.path.join(os.getcwd(), "_run")
APP_DIR = os.path.join(APP_PARENT, "streamlit_app")

if os.path.exists(APP_DIR):
    shutil.rmtree(APP_DIR)
os.makedirs(APP_PARENT, exist_ok=True)
src_folder, src_zip = os.path.join(DRIVE_DIR, "streamlit_app"), os.path.join(DRIVE_DIR, "streamlit_app.zip")
if os.path.exists(os.path.join(src_folder, "app.py")):
    shutil.copytree(src_folder, APP_DIR)
    print("Drive'daki streamlit_app klasörü kopyalandı.")
elif os.path.exists(src_zip):
    with zipfile.ZipFile(src_zip) as z:
        z.extractall(APP_PARENT)                      # zip içinde 'streamlit_app/' klasörü var
    print("streamlit_app.zip açıldı.")
else:
    raise FileNotFoundError(f"{DRIVE_DIR} içinde 'streamlit_app' klasörü veya 'streamlit_app.zip' bulunamadı.")

need = ["app.py", "app_logic.py", "rul_utils.py", "final_model.joblib", "requirements.txt", "data/test_FD001.txt", "data/RUL_FD001.txt"]
missing = [f for f in need if not os.path.exists(os.path.join(APP_DIR, f))]
print("Eksik dosya:", missing or "yok ✓")
if missing:
    raise FileNotFoundError(missing)

### 2. Streamlit'i kur
Colab'da scikit-learn 1.6.1 ve LightGBM 4.6.0 zaten yüklü (model bu ortamda eğitildi), bu yüzden yalnızca Streamlit kuruluyor.

In [ ]:
%pip install -q streamlit

In [ ]:
import sklearn, lightgbm, streamlit
print("scikit-learn", sklearn.__version__, "| lightgbm", lightgbm.__version__, "| streamlit", streamlit.__version__)
if sklearn.__version__ != "1.6.1" or lightgbm.__version__ != "4.6.0":
    print("⚠️ Sürümler modelin kaydedildiği ortamdan farklı. Model yüklenirken uyarı/hata çıkarsa şu komutu çalıştırıp "
          "Çalışma zamanı → Oturumu yeniden başlat yapın:\n   %pip install -q scikit-learn==1.6.1 lightgbm==4.6.0")

### 3. Gerçek modelle uygulama kodunun testi
Uygulamanın kullandığı fonksiyonlarla (`app_logic.py`) 100 test motoru için tahmin yapılır. Sonuç Bölüm 5'teki test RMSE ile **aynı** çıkmalı; aynıysa model dosyası, özellik dönüşümü ve uygulama kodu tutarlı demektir. Ardından örnek dosyaların doğrulama sonuçları kontrol edilir.

In [ ]:
import numpy as np, pandas as pd, joblib, json, importlib
os.chdir(APP_DIR); sys.path.insert(0, APP_DIR)
import app_logic; importlib.reload(app_logic)
from app_logic import *
from rul_utils import INDEX_COLS, FEATURES_14

model = joblib.load("final_model.joblib")
meta, cfg = load_meta()
test, rul = load_demo_test()
# Bölüm 5 ile birebir aynı hesap: her motorun son döngüsü, alttan 0'da kırpılmış tahmin
last = test.groupby("unit").tail(1).index
pred = np.clip(model.predict(test[INDEX_COLS + FEATURES_14]), 0, None)[last]
rmse = float(np.sqrt(np.mean((pred - rul.to_numpy()) ** 2)))
print(f"Model türü: {model.regressor_.named_steps['model'].__class__.__name__}")
print(f"Uygulama kodu ile test RMSE : {rmse:.3f}")
print(f"Bölüm 5'teki test RMSE      : {cfg['test_RMSE']:.3f}")
print("Tutarlı mı?", "EVET ✓ — model, özellik dönüşümü ve uygulama kodu Bölüm 5 ile birebir aynı sonucu veriyor" if abs(rmse - cfg["test_RMSE"]) < 0.01
      else "HAYIR ✗ — model dosyası Bölüm 5'te kullanılandan farklı olabilir (Drive'daki final_model.joblib'i kontrol edin)")

TR = load_train_ranges()
print("\nÖrnek dosyaların doğrulama sonuçları:")
for fn in sorted(os.listdir("ornek_veriler")):
    raw, msgs = read_uploaded(open(f"ornek_veriler/{fn}", "rb").read(), fn)
    clean, iss = (None, []) if raw is None else validate(raw, TR, meta["window"])
    levels = [m[0] for m in msgs + iss]
    durum = "REDDEDİLDİ (beklenen)" if clean is None else f"{clean['unit'].nunique()} motor tahmin edildi"
    print(f"  {fn:36s} → {durum:28s} | uyarı: {levels.count('warning')}, hata: {levels.count('error')}")
    if clean is not None:
        s, _ = summarize_engines(model, clean, window=meta["window"], cap=meta["cap"], train_ranges=TR)
        print("     ", ", ".join(f"motor {int(r.motor)}: {r.tahmini_RUL:.1f} ({r.risk})" for r in s.itertuples()))

### 4. Streamlit'i arka planda başlat

In [ ]:
import subprocess, time, urllib.request
LOG = "/content/streamlit.log" if os.path.exists("/content") else "streamlit.log"
try:
    st_proc.terminate()
except NameError:
    pass
st_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
    cwd=APP_DIR, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for i in range(40):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read().decode().strip() == "ok":
            print(f"Streamlit çalışıyor ✓ ({i+1} sn)"); break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit başlatılamadı. Log:"); print(open(LOG).read()[-3000:])

### 5. Genel bağlantı oluştur (Cloudflare tüneli)
Aşağıdaki hücre bir `https://….trycloudflare.com` linki üretir. Linke tıklayınca uygulama yeni sekmede açılır. Şifre veya hesap gerekmez.

In [ ]:
import re
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
try:
    cf_proc.terminate()
except NameError:
    pass
cf_log = "/content/cloudflared.log"
cf_proc = subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                           stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(cf_log).read())
    if m:
        url = m.group(0); break
if url:
    print("🚀 Uygulama linki:", url)
    print("(Link açılınca ilk yükleme 10–20 sn sürebilir. Colab oturumu kapanınca link de kapanır.)")
else:
    print("Tünel linki alınamadı. Aşağıdaki 'Alternatif' hücresini deneyin. Log:"); print(open(cf_log).read()[-2000:])

#### Alternatif: localtunnel (Cloudflare çalışmazsa)
Çıkan `https://….loca.lt` linkini açınca bir **"Tunnel Password"** sorulur; şifre, aşağıda yazdırılan IP adresidir.

In [ ]:
print("Tunnel Password (IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx --yes localtunnel --port 8501

### 6. Sorun giderme: Streamlit logları
Arayüzde hata görürseniz bu hücrenin çıktısını gönderin.

In [ ]:
print(open(LOG).read()[-4000:])

### 7. Durdur

In [ ]:
for p in ["cf_proc", "st_proc"]:
    try:
        globals()[p].terminate(); print(p, "durduruldu")
    except Exception:
        pass